# 🇰🇭 VoxCPM 2 Khmer Voice Cloning Cloud GPU Server
### Clean Zero-Shot Speech Synthesis (No Prompt Text Required!)

This notebook runs the backend AI inference engine for the **Khmer Voice Cloning Studio**.
Optimized for pure acoustic timbre cloning to eliminate dual-speaker hallucination / voice overlapping.

---

## ⚡ Step 1: Verify Free T4 GPU Acceleration

In [ ]:
!nvidia-smi

## 📦 Step 2: Install Required AI & Server Packages

In [ ]:
# Install server & audio packages
!pip install -q fastapi "uvicorn[standard]" python-multipart pyngrok nest_asyncio soundfile torchaudio accelerate

# Install VoxCPM library
!pip install -q voxcpm || pip install -q git+https://github.com/OpenBMB/VoxCPM.git

## 🔑 Step 3: Enter ngrok Authtoken & Launch Live Server
> **Note:** Get your free ngrok token from [https://dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)

In [ ]:
import os
import io
import gc
import shutil
import tempfile
import traceback
from typing import Optional
import torch
import torchaudio
import soundfile as sf
import nest_asyncio
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form, HTTPException, Response
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok

# --- 1. CONFIGURATION ---
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"
MODEL_NAME = "sumnim/VoxCPM2-Khmer"  # or 'openbmb/VoxCPM2'
PORT = 8000

nest_asyncio.apply()

# --- 2. FASTAPI APP INITIALIZATION ---
app = FastAPI(title="VoxCPM 2 Khmer Voice Server")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ModelHolder:
    def __init__(self):
        self.model = None
        self.sample_rate = 24000
        self.model_name = MODEL_NAME
        self.is_ready = False
        self.load_error = None
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

holder = ModelHolder()

# --- 3. MODEL LOADER (VRAM OPTIMIZED) ---
def load_model():
    print(f"\n[*] Loading VoxCPM 2 model on {holder.device.upper()} (optimize=False, load_denoiser=False)...")
    try:
        from voxcpm import VoxCPM
        try:
            holder.model = VoxCPM.from_pretrained(MODEL_NAME, load_denoiser=False, optimize=False)
        except Exception as e:
            print(f"[!] Khmer specific weights '{MODEL_NAME}' falling back to 'openbmb/VoxCPM2': {e}")
            holder.model_name = "openbmb/VoxCPM2"
            holder.model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False, optimize=False)
        
        if hasattr(holder.model, 'tts_model') and hasattr(holder.model.tts_model, 'sample_rate'):
            holder.sample_rate = holder.model.tts_model.sample_rate
        else:
            holder.sample_rate = 24000
            
        holder.is_ready = True
        print(f"[✓] VoxCPM 2 successfully loaded (Sample Rate: {holder.sample_rate}Hz)!")
    except Exception as e:
        holder.load_error = str(e)
        print(f"[!] Failed to load model: {e}")

# --- 4. ENDPOINTS ---
@app.get("/health")
def health():
    cuda = torch.cuda.is_available()
    vram_alloc = round(torch.cuda.memory_allocated(0)/(1024**3), 2) if cuda else 0.0
    vram_total = round(torch.cuda.get_device_properties(0).total_memory/(1024**3), 2) if cuda else 0.0
    return {
        "status": "ready" if holder.is_ready else ("loading" if not holder.load_error else "error"),
        "model_name": holder.model_name,
        "device": holder.device,
        "cuda_available": cuda,
        "gpu_name": torch.cuda.get_device_name(0) if cuda else "CPU",
        "vram_allocated_gb": vram_alloc,
        "vram_total_gb": vram_total,
        "sample_rate": holder.sample_rate,
        "error": holder.load_error
    }

@app.post("/clone-voice")
async def clone_voice(
    text: str = Form(...),
    prompt_text: Optional[str] = Form(None),
    audio_file: UploadFile = File(...),
    inference_timesteps: int = Form(8),
    cfg_scale: float = Form(2.0)
):
    if not text.strip():
        raise HTTPException(400, "Text is required.")
    
    clean_prompt = prompt_text.strip() if prompt_text and prompt_text.strip() else None
    temp_dir = tempfile.mkdtemp()
    temp_ref_path = os.path.join(temp_dir, "ref.wav")
    
    try:
        with open(temp_ref_path, "wb") as f:
            f.write(await audio_file.read())
            
        try:
            w, sr = torchaudio.load(temp_ref_path)
            if w.shape[0] > 1:
                w = torch.mean(w, dim=0, keepdim=True)
            if sr != 24000:
                w = torchaudio.transforms.Resample(sr, 24000)(w)
                sr = 24000
            torchaudio.save(temp_ref_path, w, sr)
        except Exception:
            pass

        if holder.model and holder.is_ready:
            with torch.inference_mode():
                gen_audio = None
                if clean_prompt is None:
                    try:
                        gen_audio = holder.model.generate(
                            text=text.strip(),
                            reference_wav_path=temp_ref_path,
                            cfg_value=float(cfg_scale),
                            inference_timesteps=int(inference_timesteps)
                        )
                    except Exception:
                        try:
                            gen_audio = holder.model.generate(
                                text=text.strip(),
                                prompt_wav_path=temp_ref_path,
                                prompt_text=None,
                                cfg_value=float(cfg_scale),
                                inference_timesteps=int(inference_timesteps)
                            )
                        except Exception:
                            gen_audio = holder.model.generate(
                                text=text.strip(),
                                prompt_wav=temp_ref_path,
                                cfg_scale=float(cfg_scale),
                                steps=int(inference_timesteps)
                            )
                else:
                    gen_audio = holder.model.generate(
                        text=text.strip(),
                        prompt_wav_path=temp_ref_path,
                        prompt_text=clean_prompt,
                        cfg_value=float(cfg_scale),
                        inference_timesteps=int(inference_timesteps)
                    )
                    
            if isinstance(gen_audio, torch.Tensor):
                gen_audio = gen_audio.detach().cpu().float().numpy()
            if len(gen_audio.shape) > 1:
                gen_audio = gen_audio.squeeze()
                
            out_buf = io.BytesIO()
            sf.write(out_buf, gen_audio, holder.sample_rate, format='WAV', subtype='PCM_16')
            wav_bytes = out_buf.getvalue()
        else:
            import numpy as np
            t = np.linspace(0, 2.0, 24000 * 2, endpoint=False)
            mock_tone = (0.2 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)
            out_buf = io.BytesIO()
            sf.write(out_buf, mock_tone, 24000, format='WAV')
            wav_bytes = out_buf.getvalue()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        return Response(
            content=wav_bytes,
            media_type="audio/wav",
            headers={"Content-Disposition": "attachment; filename=khmer_cloned_voice.wav"}
        )
    finally:
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir, ignore_errors=True)

# --- 5. LAUNCH NGROK & RUN ---
if NGROK_AUTH_TOKEN and NGROK_AUTH_TOKEN != "YOUR_NGROK_AUTHTOKEN_HERE":
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
try:
    ngrok.kill()
except:
    pass

tunnel = ngrok.connect(PORT, proto="http")
public_url = tunnel.public_url.replace("http://", "https://")

load_model()

print("\n" + "="*65)
print(f"🔥 PUBLIC API URL: {public_url}")
print("Copy & paste this URL into your Next.js Web App API Settings!")
print("="*65 + "\n")

uvicorn.run(app, host="0.0.0.0", port=PORT)